# Utils - QB - FrequencyConverter

Ce notebook illustre et vérifie le comportement de la classe `FrequencyConverter`
(`tsforecast/utils/frequency/converter.py`), moteur générique de conversion de
fréquence utilisé dans tout le package (agrégation, interpolation, alignement de
fréquences mixtes). Pour la construction de jeux d'entraînement/prédiction dont
l'index d'origine doit être préservé ou densifié, voir `FrequencyAligner`
(`tsforecast/frequency/frequency_aligner.py`), qui délègue ses conversions à
cette classe.

**Périmètre** : seules les méthodes publiques de `FrequencyConverter` sont testées :
- `convert(value, from_unit, to_unit, **kwargs)`
- `convert_frequency(data, target_freq, method, alignment_method, time_col, panel_cols, target_position, full_periods_only, limit, limit_direction, limit_area)`
- `get_conversion_factor(from_unit, to_unit)`
- `count_subperiods_per_period(target_index, low_freq, high_freq)`
- `aggregate_to_lower_frequency(data, target_freq, method, full_periods_only)`
- `interpolate_to_higher_frequency(data, target_freq, method, limit, limit_direction, limit_area, source_freq)`

Les méthodes privées (préfixées `_`, ex. `_upsample`, `_downsample`,
`_extend_index_for_upsampling`, `_align_mixed_frequency_columns`...) ne sont pas
testées directement : elles sont couvertes indirectement via les méthodes
publiques qui les appellent.

## Correctifs appliqués avant ce notebook

Deux bugs bloquants ont été identifiés en préparant ce notebook et corrigés dans
`tsforecast/utils/frequency/converter.py` (sans eux, une large part des
illustrations ci-dessous aurait été impossible) :

1. **`get_conversion_factor()`** référençait des noms non définis (`to_freq` /
   `from_freq` au lieu des paramètres réels `from_unit` / `to_unit`) : tout appel
   levait un `NameError`. Correction : normalisation de `from_unit`/`to_unit` vers
   leur base de durée (`normalize_frequency(..., return_format='base')`) puis
   délégation à `get_duration_conversion_factor(to_base, from_base)`.
2. **`interpolate_to_higher_frequency()`** (et donc `convert_frequency()` en
   upsampling) levait un `AttributeError` dès que la fréquence source et la
   fréquence cible avaient des bases différentes (ex. trimestriel → mensuel, le
   cas d'upsampling le plus courant) : `_extend_index_for_upsampling` appelait
   `self._duration_converter`, un attribut jamais initialisé par `__init__`.
   Correction : remplacement par un appel direct à la fonction utilitaire
   `get_duration_conversion_factor()` (déjà importée dans le module), sans
   instancier de `DurationConverter` dédié.

Les deux correctifs ont été vérifiés par la suite de tests existante
(`tests/frequency/test_converter_*.py`) : aucune régression, et un test
auparavant en échec (`test_falls_back_on_constant_count`) passe désormais.

**Jeux de données** : on réutilise les jeux de données créés dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb` (`df_timeseries`,
`df_panel` : indicateurs macroéconomiques mensuels/trimestriels/annuels, délais
de publication, couverture temporelle hétérogène par pays, fréquence de
publication hétérogène pour `depenses_publiques_pib`) comme données réalistes,
complétés par de petits jeux synthétiques ciblés pour couvrir des cas que ces
jeux ne couvrent pas naturellement (données journalières, positions S/E
explicites, trous de NaN internes/aux extrémités, panel trimestriel à profils de
NaN asymétriques).

**Remarque** : un premier notebook d'exploration plus succinct existait dans
`0 - QB - Utils.ipynb` ; les classes/fonctions ayant été refactorées depuis, ses
API ne sont plus toutes à jour, mais il a servi d'inspiration pour certains cas
de test ci-dessous.

Ce notebook a vocation à servir de base à de futurs tests unitaires
(`tests/utils/frequency/test_converter.py`, qui n'existe pas encore — à ne pas
confondre avec les tests existants de `tests/frequency/test_converter_*.py`,
organisés par thème plutôt que par méthode).

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings
from typing import get_args

import numpy as np
import pandas as pd

# Classe testée
from tsforecast.utils.frequency.converter import FrequencyConverter

# Types exportés (pour lister les fréquences supportées, à titre indicatif)
from tsforecast.utils.frequency.normalizer import FrequencyType, UserFrequencyType

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

# Instanciation du convertisseur
converter = FrequencyConverter()

print("Fréquences (codes) déclarées dans le type FrequencyType :")
print(get_args(FrequencyType))
print()
print("Fréquences (littéraux) déclarées dans le type UserFrequencyType :")
print(get_args(UserFrequencyType))

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (aucune fonction
partagée n'existe entre notebooks dans ce projet) pour obtenir `df_timeseries`
(séries temporelles macroéconomiques) et `df_panel` (panel France/Allemagne/Italie
à couverture et fréquences hétérogènes).

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle
    df['balance_commerciale_annuelle'] = np.nan
    for i, date in enumerate(dates):
        if date.month == 1:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Simulation des délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Simulation de données historiques limitées
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


df_timeseries = create_timeseries_dataset()
print(f"df_timeseries : {df_timeseries.shape}, {df_timeseries.index.min().date()} -> {df_timeseries.index.max().date()}")
df_timeseries.tail()

In [ ]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        # Production industrielle
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        # Inflation
        infl_trend = np.linspace(
            params['inflation_base'],
            params['inflation_base'] + np.random.uniform(0.5, 2.0),
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        # Taux de chômage
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Balance commerciale annuelle
        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 1:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        publication_months = [1] if params['depenses_frequency'] == 'annuelle' else [1, 4, 7, 10]
        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan
        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


df_panel = create_panel_dataset()
print(f"df_panel : {df_panel.shape}, entités : {df_panel.index.get_level_values('country').unique().tolist()}")
for country in df_panel.index.get_level_values('country').unique():
    dates_country = df_panel.loc[country].index
    print(f"  {country:12s} {dates_country.min().strftime('%Y-%m')} -> {dates_country.max().strftime('%Y-%m')}")
df_panel.loc['France'].tail()

### 2.2 - Jeux de données synthétiques complémentaires

`df_timeseries`/`df_panel` sont uniquement mensuels/trimestriels/annuels et ne
couvrent ni les données journalières, ni les positions explicites S/E, ni des
motifs de NaN ciblés (trous internes, NaN symétriques en début/fin). Ces jeux
synthétiques comblent ces cas.

In [ ]:
# Série journalière (marche aléatoire) : pour tester les conversions D -> W/M/Q/Y
rng = np.random.default_rng(0)
dates_daily = pd.date_range('2023-01-01', periods=730, freq='D')
serie_journaliere = pd.Series(
    100 + np.cumsum(rng.normal(0, 1, len(dates_daily))),
    index=dates_daily, name='indice_synthetique'
)

# Séries trimestrielles à position explicite S (QS) et E (QE)
dates_qs = pd.date_range('2020-01-01', periods=12, freq='QS')
serie_qs = pd.Series(np.linspace(100, 155, 12), index=dates_qs, name='pib_qs')
dates_qe = pd.date_range('2020-03-31', periods=12, freq='QE')
serie_qe = pd.Series(np.linspace(100, 155, 12), index=dates_qe, name='pib_qe')

# Série trimestrielle avec NaN symétriques en début (historique manquant)
# et en fin (délais de publication) : scénario réaliste
dates_q_nan = pd.date_range('2020-01-01', periods=10, freq='QS')
serie_nan_start_end = pd.Series(
    [np.nan, np.nan, 105, 110, 115, 120, np.nan, 130, np.nan, np.nan],
    index=dates_q_nan, name='pib_delays'
)

# Série trimestrielle avec un trou de NaN interne (pour limit_area)
dates_q_hole = pd.date_range('2020-01-01', periods=8, freq='QS')
serie_hole = pd.Series(
    [100, np.nan, 115, 130, 125, np.nan, 135, 150],
    index=dates_q_hole, name='pib_hole'
)

# Série mensuelle de 14 mois : le dernier trimestre est incomplet (2/3 mois)
dates_m_14 = pd.date_range('2024-01-01', periods=14, freq='MS')
serie_incomplete_quarter = pd.Series(range(100, 114), index=dates_m_14, name='indicateur', dtype=float)

# DataFrame journalier à 2 colonnes, pour tester target_freq en dict
dates_mixed = pd.date_range('2023-01-01', periods=180, freq='D')
rng2 = np.random.default_rng(1)
df_mixte = pd.DataFrame({
    'ventes_jour': rng2.normal(100, 10, 180),
    'temperature': np.sin(np.linspace(0, 2 * np.pi, 180)) * 15 + 20,
}, index=dates_mixed)

# Panel trimestriel synthétique à profils de NaN asymétriques (pour interpolate)
dates_pq = pd.date_range('2020-01-01', periods=8, freq='QS')
panel_q = pd.concat([
    pd.DataFrame({'pib': [100, 110, 105, 120, 115, 130, np.nan, np.nan], 'entity': 'A'}, index=dates_pq),
    pd.DataFrame({'pib': [np.nan, np.nan, 200, 210, 205, 220, 215, 230], 'entity': 'B'}, index=dates_pq),
]).set_index('entity', append=True).swaplevel()
panel_q.index.names = ['entity', 'date']

print(f"serie_journaliere       : {len(serie_journaliere)} points")
print(f"serie_qs / serie_qe     : {len(serie_qs)} / {len(serie_qe)} points")
print(f"serie_nan_start_end     : {serie_nan_start_end.isna().sum()} NaN / {len(serie_nan_start_end)}")
print(f"serie_hole              : {serie_hole.isna().sum()} NaN / {len(serie_hole)}")
print(f"serie_incomplete_quarter: {len(serie_incomplete_quarter)} points (dernier trimestre incomplet)")
print(f"df_mixte                : {df_mixte.shape}")
print(f"panel_q                 : {panel_q.shape}")

## 3 - `get_conversion_factor()`

Facteur de conversion approximatif entre deux fréquences, utilisé en interne
pour résoudre la valeur `'default'` de `limit` (interpolation) et comme repli
constant de `count_subperiods_per_period()`.

In [ ]:
# Facteurs de référence entre fréquences courantes
paires = [
    ('daily', 'monthly'), ('monthly', 'quarterly'), ('quarterly', 'annual'),
    ('daily', 'weekly'), ('monthly', 'annual'),
]
for a, b in paires:
    print(f"{a:>10s} -> {b:<10s} : {converter.get_conversion_factor(a, b):.4f}")

In [ ]:
# Symétrie : factor(a, b) * factor(b, a) == 1 (c'est un simple ratio de durées)
for a, b in paires:
    fab = converter.get_conversion_factor(a, b)
    fba = converter.get_conversion_factor(b, a)
    assert np.isclose(fab * fba, 1.0), (a, b, fab, fba)
print("OK : get_conversion_factor(a, b) * get_conversion_factor(b, a) == 1")

# Identité
for u in ['D', 'M', 'Q', 'Y', 'W']:
    assert converter.get_conversion_factor(u, u) == 1.0
print("OK : get_conversion_factor(u, u) == 1.0 pour toutes les fréquences testées")

In [ ]:
# Formats mixtes : codes pandas ('D', 'M') et littéraux ('daily', 'monthly')
# sont interchangeables et peuvent être mélangés entre from_unit et to_unit
resultats = {
    "code -> code": converter.get_conversion_factor('D', 'M'),
    "littéral -> littéral": converter.get_conversion_factor('daily', 'monthly'),
    "code -> littéral": converter.get_conversion_factor('D', 'monthly'),
    "littéral -> code": converter.get_conversion_factor('daily', 'M'),
}
for label, valeur in resultats.items():
    print(f"{label:25s} -> {valeur}")
assert len(set(resultats.values())) == 1, "Les 4 formats devraient donner le même résultat" 

In [ ]:
# Erreurs : fréquences non supportées
for a, b in [('foo', 'D'), ('D', 'foo')]:
    try:
        converter.get_conversion_factor(a, b)
        print(f"get_conversion_factor({a!r}, {b!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"get_conversion_factor({a!r}, {b!r}) -> ValueError : {e}")

In [ ]:
# Application aux délais de publication du notebook 3 (nombre de périodes équivalentes)
indicateurs = {
    "pib_trimestriel": {"frequence": "Q", "delai": (2, "M")},
    "inflation_ipc": {"frequence": "M", "delai": (1, "M")},
    "balance_commerciale_annuelle": {"frequence": "Y", "delai": (3, "M")},
}
for nom, infos in indicateurs.items():
    n, unite = infos["delai"]
    freq = infos["frequence"]
    facteur = converter.get_conversion_factor(unite, freq)
    print(f"{nom:32s} délai={n} {unite} -> {n * facteur:.2f} période(s) de {freq}")

## 4 - `count_subperiods_per_period()`

Compte le nombre exact de sous-périodes (`high_freq`) contenues dans chaque
période cible (`low_freq`) d'un index donné, en s'appuyant sur les `Period`
pandas concrets plutôt que sur un facteur constant : février compte 28 jours,
janvier 31, là où `get_conversion_factor('D', 'M')` renverrait ~30.4 pour les
deux. Un repli sur ce facteur constant s'applique quand la base de fréquence
n'est pas convertible en `Period` pandas.

In [ ]:
# Comptage exact : le nombre de jours par mois varie (28-31)
idx_m = pd.date_range('2023-01-31', periods=12, freq='ME')
counts_jours = converter.count_subperiods_per_period(idx_m, 'M', 'D')
print(dict(zip(idx_m.strftime('%Y-%m'), counts_jours)))

# Comptage exact : nombre de jours par an, y compris année bissextile (2020)
idx_y = pd.date_range('2020-12-31', periods=3, freq='YE')
counts_annee = converter.count_subperiods_per_period(idx_y, 'Y', 'D')
print(dict(zip(idx_y.year, counts_annee)))
assert counts_annee[0] == 366  # 2020 est bissextile
assert (counts_annee[1:] == 365).all()

In [ ]:
# Écart avec le facteur constant de get_conversion_factor : 365.0 pour toutes
# les années, alors que count_subperiods_per_period distingue 2020 (366 j.)
facteur_constant = converter.get_conversion_factor('D', 'Y')
print(f"Facteur constant (D -> Y) : {facteur_constant}")
print(f"Comptage exact 2020/2021/2022 : {counts_annee.tolist()}")
assert counts_annee[0] != facteur_constant  # l'année bissextile diffère du facteur constant

In [ ]:
# Repli sur le comptage constant quand la fréquence n'est pas convertible en
# Period pandas ('SM', semi-mensuel, n'a pas d'équivalent Period)
idx_repli = pd.date_range('2023-01-31', periods=2, freq='ME')
r_repli = converter.count_subperiods_per_period(idx_repli, 'M', 'SM')
print("repli constant (M -> SM) :", r_repli, "== get_conversion_factor('SM', 'M') :", converter.get_conversion_factor('SM', 'M'))
assert (r_repli == converter.get_conversion_factor('SM', 'M')).all()

In [ ]:
# Application sur les données réelles : nombre de jours par mois sur la fin
# de df_timeseries (utile pour pondérer une agrégation D -> M par le nombre
# de jours effectivement couverts par chaque mois)
idx_prod = df_timeseries.index[-6:]
print(dict(zip(idx_prod.strftime('%Y-%m'), converter.count_subperiods_per_period(idx_prod, 'M', 'D'))))

## 5 - `convert()` : point d'entrée générique

`convert(value, from_unit, to_unit, **kwargs)` est l'implémentation de la
méthode abstraite `TemporalConverter.convert()`. Elle redirige directement vers
`convert_frequency(data=value, target_freq=to_unit, **kwargs)` : `from_unit`
n'est **jamais utilisé**, la fréquence source est toujours auto-détectée depuis
l'index.

In [ ]:
# convert() et convert_frequency() sont strictement équivalents
r_convert = converter.convert(serie_journaliere, from_unit='daily', to_unit='monthly', method='mean')
r_convert_frequency = converter.convert_frequency(serie_journaliere, target_freq='monthly', method='mean')
assert r_convert.equals(r_convert_frequency)
print("OK : convert(value, from_unit, to_unit) == convert_frequency(data=value, target_freq=to_unit)")

# from_unit est ignoré : une valeur farfelue ne change strictement rien au résultat
r_from_unit_ignore = converter.convert(serie_journaliere, from_unit='XYZZY_NON_VALIDE', to_unit='monthly', method='mean')
assert r_from_unit_ignore.equals(r_convert)
print("OK : from_unit n'est jamais validé ni utilisé, seule la détection automatique compte")

In [ ]:
# DataFrame
r_df = converter.convert(df_mixte, from_unit='daily', to_unit='weekly', method='mean')
print(r_df.head())

# Panel (MultiIndex)
r_panel = converter.convert(df_panel[['production_industrielle']], from_unit='monthly', to_unit='quarterly', method='mean')
print(r_panel.head())

## 6 - `convert_frequency()` : méthode principale

Détecte automatiquement la fréquence source, choisit la direction (upsampling
via `interpolate_to_higher_frequency` / downsampling via
`aggregate_to_lower_frequency`), et gère position, alignement multi-colonnes et
panel.

### 6.1 - Downsampling et upsampling simples (Series)

In [ ]:
# Downsampling : mensuel -> trimestriel (aggregate_to_lower_frequency en interne)
serie_prod_q = converter.convert_frequency(df_timeseries['production_industrielle'].dropna(), 'quarterly', method='mean')
print("Downsampling mensuel -> trimestriel :")
print(serie_prod_q.head())

# Upsampling : trimestriel -> mensuel (interpolate_to_higher_frequency en
# interne) - fonctionne désormais grâce au correctif de _extend_index_for_upsampling
serie_pib = df_timeseries['pib_trimestriel'].dropna()
serie_pib_m = converter.convert_frequency(serie_pib, 'monthly', method='linear')
print("\nUpsampling trimestriel -> mensuel :")
print(serie_pib_m.head(6))

### 6.2 - `target_freq` avec position explicite (`QS` vs `QE`)

In [ ]:
for tf in ['QS', 'QE']:
    r = converter.convert_frequency(df_timeseries['production_industrielle'].dropna(), tf, method='mean')
    print(f"target_freq={tf!r:6s} -> premières dates : {[d.date() for d in r.index[:3]]}")

### 6.3 - `target_position` explicite vs `None`

D'après la docstring, `target_position=None` devrait « préserver la position
source quand elle est identifiable ». En pratique, pour une Series, le code
décompose la **chaîne `target_freq` fournie par l'utilisateur** (pas la position
de la source) :

```python
if target_position is None:
    target_freq_base, target_position, _ = normalize_frequency(target_freq, return_format='components')
```

Si `target_freq` est une base nue sans suffixe S/E (ex. `'M'`), sa position
décomposée est `None`, et la fréquence finale retombe alors sur la convention
pandas historique des bases nues (fin de période, ex. `'ME'`) — **indépendamment
de la position de la source**. Pour obtenir la position de la source, il faut
soit passer un `target_freq` déjà suffixé (`'MS'`/`'ME'`), soit fixer
`target_position` explicitement. C'est un point de vigilance par rapport au
comportement documenté.

In [ ]:
# Série source en position S (QS) ; target_freq='M' est une base nue sans
# suffixe explicite
serie_pib_qs = df_timeseries['pib_trimestriel'].dropna()
print("Position détectée de la source :", serie_pib_qs.index.freqstr if serie_pib_qs.index.freq else "non fixée sur l'objet, mais QS d'après la construction")

r_none = converter.convert_frequency(serie_pib_qs, 'M', method='linear', target_position=None)
r_s = converter.convert_frequency(serie_pib_qs, 'M', method='linear', target_position='S')
r_e = converter.convert_frequency(serie_pib_qs, 'M', method='linear', target_position='E')

print("target_position=None :", [d.date() for d in r_none.index[:3]], "(retombe sur 'E', PAS la position S de la source)")
print("target_position='S'  :", [d.date() for d in r_s.index[:3]])
print("target_position='E'  :", [d.date() for d in r_e.index[:3]])
assert r_none.index.equals(r_e.index), "target_position=None retombe ici sur la position 'E', pas sur celle de la source" 

### 6.4 - DataFrame avec `target_freq` en dictionnaire (fréquences mixtes)

In [ ]:
# ventes_jour (journalier) -> mensuel, temperature (journalier) -> hebdomadaire
# L'index de sortie est l'UNION des index cibles : l'index journalier d'origine
# disparaît dès que toutes les colonnes sont converties
r_dict = converter.convert_frequency(
    df_mixte,
    target_freq={'ventes_jour': 'monthly', 'temperature': 'weekly'},
    method='mean',
    alignment_method='ffill',
)
print(f"{len(df_mixte)} lignes journalières -> {len(r_dict)} lignes (union mensuel ∪ hebdomadaire)")
r_dict.head(10)

### 6.5 - `alignment_method` : `ffill` / `bfill` / `nearest` / `none`

In [ ]:
for am in ['ffill', 'bfill', 'nearest', 'none']:
    r = converter.convert_frequency(
        df_mixte, target_freq={'ventes_jour': 'monthly', 'temperature': 'weekly'},
        method='mean', alignment_method=am,
    )
    print(f"alignment_method={am:6s} : {r['ventes_jour'].isna().sum():2d} NaN (ventes_jour), {r['temperature'].isna().sum():2d} NaN (temperature)")
# 'none' laisse le plus de NaN (aucun remplissage) ; 'ffill'/'bfill'/'nearest' en comblent une partie

### 6.6 - Panel : conversion entité par entité (`df_panel`)

In [ ]:
# Chaque entité est convertie indépendamment sur son propre index simple :
# la couverture temporelle hétérogène (France/Allemagne/Italie) est préservée
r_panel_q = converter.convert_frequency(df_panel[['production_industrielle', 'inflation_ipc']], target_freq='quarterly', method='mean')
print("France (premières lignes) :")
print(r_panel_q.loc['France'].head())
print("\nAllemagne (premières lignes, début de couverture différent) :")
print(r_panel_q.loc['Allemagne'].head())

In [ ]:
# depenses_publiques_pib est publiée annuellement pour FR/IT mais trimestriellement
# pour l'Allemagne : la conversion vers 'annual' agrège chaque entité depuis SA
# propre fréquence source, détectée indépendamment
r_depenses = converter.convert_frequency(df_panel[['depenses_publiques_pib']], target_freq='annual', method='mean')
for country in ['France', 'Allemagne', 'Italie']:
    print(f"\n{country} :")
    print(r_depenses.loc[country].dropna())

### 6.7 - `target_freq` en dictionnaire avec clés d'entité (spécificité par pays)

In [ ]:
# Clé (entité,) : cible propre à une entité. Les entités absentes du dict
# (ici Italie) conservent leur fréquence d'origine (mensuelle), inchangée.
r_panel_dict = converter.convert_frequency(
    df_panel[['production_industrielle']],
    target_freq={('France',): 'quarterly', ('Allemagne',): 'annual'},
    method='mean',
)
print("France (trimestriel) :")
print(r_panel_dict.loc['France'].head(3))
print("\nAllemagne (annuel) :")
print(r_panel_dict.loc['Allemagne'].head(3))
print("\nItalie (non ciblée -> mensuel inchangé) :")
print(r_panel_dict.loc['Italie'].head(3))

### 6.8 - Cas particulier et erreurs

In [ ]:
# Fréquence source == cible (base + position EXACTEMENT identiques) : les
# données sont retournées TELLES QUELLES, sans passer par aggregate/interpolate.
# Attention : il faut que target_freq porte la même position que la source
# (ici 'MS') -- le littéral bare 'monthly' ne suffit PAS (cf. 6.3 : il retombe
# sur la position 'E' et re-resample silencieusement vers 'ME').
serie_inflation = df_timeseries['inflation_ipc'].dropna()
print("Position de la source :", serie_inflation.index.freqstr)

r_same = converter.convert_frequency(serie_inflation, 'MS', method='mean')
assert r_same.equals(serie_inflation)
print("OK : target_freq='MS' (position identique à la source) -> données inchangées")

r_bare = converter.convert_frequency(serie_inflation, 'monthly', method='mean')
assert not r_bare.equals(serie_inflation)
print("target_freq='monthly' (bare, sans position) -> RE-RESAMPLE vers 'ME' :", [d.date() for d in r_bare.index[:3]])

# target_freq vide
try:
    converter.convert_frequency(serie_inflation, '')
except ValueError as e:
    print("target_freq='' -> ValueError :", e)

# dict target_freq sur une Series non-panel : invalide
try:
    converter.convert_frequency(serie_pib, {'a': 'M'})
except ValueError as e:
    print("dict target_freq sur Series non-panel -> ValueError :", e)

## 7 - `aggregate_to_lower_frequency()`

Agrégation (downsampling) via `resample`. Explore `method` (9 agrégations
numériques + `'all'`/`'any'` booléens) et `full_periods_only`.

### 7.1 - Méthodes d'agrégation numériques

In [ ]:
dates_m12 = pd.date_range('2024-01-01', periods=12, freq='MS')
values_12 = [100, 110, 105, 120, 115, 125, 130, 140, 135, 150, 145, 155]
serie_m12 = pd.Series(values_12, index=dates_m12, dtype=float, name='indicateur')

for method in ['mean', 'sum', 'first', 'last', 'min', 'max', 'median', 'std', 'count']:
    r = converter.aggregate_to_lower_frequency(serie_m12, 'QS', method=method)
    print(f"{method:>6s} : {r.tolist()}")

### 7.2 - `full_periods_only` : masquage des périodes incomplètes

`serie_incomplete_quarter` couvre 14 mois (janvier 2024 -> février 2025) : le
premier trimestre de 2025 ne contient que 2 des 3 mois attendus.

In [ ]:
for fpo in [False, True]:
    print(f"full_periods_only={fpo} :")
    for method in ['mean', 'sum', 'count']:
        r = converter.aggregate_to_lower_frequency(serie_incomplete_quarter, 'QS', method=method, full_periods_only=fpo)
        print(f"  {method:>6s} : {r.tolist()}")
# full_periods_only=False : le dernier trimestre est agrégé sur ses 2 mois disponibles (sum SOUS-ESTIMÉE)
# full_periods_only=True  : le dernier trimestre (incomplet) devient NaN

### 7.3 - Méthodes `'all'` / `'any'` : sémantique booléenne, indépendante de `full_periods_only`

`'all'` est vrai ssi TOUTES les sous-périodes attendues sont présentes ET
vraies (une période partiellement couverte en bord de grille est toujours
fausse, même si les valeurs présentes sont toutes vraies). `'any'` est vrai dès
qu'au moins une valeur présente est vraie.

In [ ]:
mask_complet = pd.Series(True, index=pd.date_range('2023-01-31', periods=12, freq='ME'))
mask_partiel = pd.Series(True, index=pd.date_range('2023-01-31', periods=7, freq='ME'))  # 7/12 mois
mask_avec_faux = mask_complet.copy()
mask_avec_faux.iloc[3] = False

print("all, 12/12 mois vrais         :", converter.aggregate_to_lower_frequency(mask_complet, 'YE', method='all').tolist())
print("all, 7/12 mois seulement      :", converter.aggregate_to_lower_frequency(mask_partiel, 'YE', method='all').tolist(), "(couverture incomplète -> False, même si les 7 valeurs présentes sont vraies)")
print("all, 12/12 mois avec 1 faux   :", converter.aggregate_to_lower_frequency(mask_avec_faux, 'YE', method='all').tolist())
print("any, 7/12 mois seulement      :", converter.aggregate_to_lower_frequency(mask_partiel, 'YE', method='any').tolist())

# full_periods_only est ignoré pour 'all'/'any' : les deux appels donnent le même résultat
r_fpo_true = converter.aggregate_to_lower_frequency(mask_partiel, 'YE', method='all', full_periods_only=True)
r_fpo_false = converter.aggregate_to_lower_frequency(mask_partiel, 'YE', method='all', full_periods_only=False)
assert r_fpo_true.equals(r_fpo_false)
print("OK : full_periods_only n'a aucun effet sur 'all'/'any' (couverture déjà encodée dans la sémantique booléenne)")

### 7.4 - Application sur `df_timeseries` : production industrielle mensuelle -> trimestrielle

In [ ]:
# 2018 (avant le début de disponibilité de la série, cf. section 2.1) donne des
# trimestres NaN avec full_periods_only=True
r_prod_q = converter.aggregate_to_lower_frequency(df_timeseries['production_industrielle'], 'QE', method='mean', full_periods_only=True)
print(r_prod_q.head(6))

## 8 - `interpolate_to_higher_frequency()`

Interpolation (upsampling). Explore `method`, `limit`, `limit_direction`,
`limit_area` et `source_freq`.

### 8.1 - Méthodes d'interpolation

In [ ]:
dates_q8 = pd.date_range('2024-01-01', periods=8, freq='QS')
serie_q8 = pd.Series([100, 120, 115, 130, 125, 140, 135, 150], index=dates_q8, dtype=float, name='pib')

for method in ['linear', 'nearest', 'cubic', 'zero', 'slinear']:
    r = converter.interpolate_to_higher_frequency(serie_q8, 'MS', method=method)
    print(f"{method:>8s} : {len(r)} points, {r.isna().sum()} NaN")

### 8.2 - `limit` : nombre de NaN consécutifs comblés

`'default'` utilise le facteur de conversion (3 pour trimestre -> mois) ; `None`
ne limite rien ; un entier fixe la limite explicitement.

In [ ]:
for lim in ['default', None, 1, 2]:
    r = converter.interpolate_to_higher_frequency(serie_q8, 'MS', method='linear', limit=lim)
    print(f"limit={str(lim):>9s} : {r.isna().sum()} NaN | extrait : {r.iloc[:6].round(2).tolist()}")
# limit='default' et limit=None coïncident ici car chaque trou ne compte que 2 NaN (< 3)

### 8.3 - `limit_direction` et lien avec la position cible

Sans direction explicite, la direction par défaut dépend de la position de la
fréquence cible : `'forward'` pour une position start (`MS`), `'backward'` pour
une position end (`ME`).

In [ ]:
for direction in ['forward', 'backward', 'both']:
    r = converter.interpolate_to_higher_frequency(serie_q8, 'MS', method='linear', limit='default', limit_direction=direction)
    print(f"limit_direction={direction:>8s} : {r.isna().sum()} NaN restants")

print()
for pos, tf in [('S', 'MS'), ('E', 'ME')]:
    r = converter.interpolate_to_higher_frequency(serie_q8, tf, method='linear', limit='default')
    print(f"target_freq={tf} (position {pos}), limit_direction non spécifié : {r.isna().sum()} NaN")
# position S -> direction implicite 'forward' (le dernier trou est comblé)
# position E -> direction implicite 'backward' (le dernier trou vers l'avant reste vide)

### 8.4 - `limit_area` : `None` / `'inside'` / `'outside'`

`serie_hole` contient un trou interne (NaN entouré de valeurs connues) ;
`'inside'` ne comble que ce type de trou (pas d'extrapolation aux extrémités),
`'outside'` fait l'inverse.

In [ ]:
for area in [None, 'inside', 'outside']:
    r = converter.interpolate_to_higher_frequency(serie_hole, 'MS', method='linear', limit=None, limit_area=area)
    print(f"limit_area={str(area):>9s} : {r.isna().sum()} NaN / {len(r)} points")

### 8.5 - Scénario réaliste : NaN en début (historique manquant) et en fin (délai de publication)

`limit_area='inside'` évite toute extrapolation hasardeuse aux extrémités —
recommandé pour des données économiques sujettes à des délais de publication.

In [ ]:
r_nan_edges = converter.interpolate_to_higher_frequency(serie_nan_start_end, 'MS', method='linear', limit=None, limit_area='inside')
print(r_nan_edges)
assert r_nan_edges.iloc[:6].isna().all()  # début : jamais comblé (limit_area='inside')
assert r_nan_edges.iloc[-6:].isna().all()  # fin (délai de publication) : jamais comblé non plus

### 8.6 - `source_freq` explicite

In [ ]:
# Utile quand l'index ne permet pas de détecter fiablement la fréquence source
# (ex. index court, ou variable portée sur un index d'une autre fréquence)
r_source_explicite = converter.interpolate_to_higher_frequency(serie_q8, 'MS', method='linear', source_freq='QS')
r_source_auto = converter.interpolate_to_higher_frequency(serie_q8, 'MS', method='linear')
assert r_source_explicite.equals(r_source_auto)
print("OK : source_freq explicite (QS) donne le même résultat que la détection automatique ici")

### 8.7 - Application réelle : PIB trimestriel -> mensuel (`df_timeseries`)

In [ ]:
# Fonctionne désormais de bout en bout grâce au correctif de _extend_index_for_upsampling
pib_mensuel = converter.interpolate_to_higher_frequency(
    df_timeseries['pib_trimestriel'].dropna(), 'MS', method='linear', limit_area='inside'
)
print(pib_mensuel.head(8))

## 9 - Panel : `aggregate_to_lower_frequency()` / `interpolate_to_higher_frequency()` appelées directement

Contrairement à `convert_frequency()`, ces deux méthodes ne gèrent PAS le
MultiIndex panel nativement : elles doivent être appliquées entité par entité
(c'est exactement ce que fait `convert_frequency()` en interne via
`_convert_panel_frequency`).

In [ ]:
# Panel synthétique à profils de NaN asymétriques (entité A : NaN en fin, entité B : NaN en début)
print(panel_q)

print("\nInterpolation Q -> M par entité, limit_area='inside' :")
for entity in ['A', 'B']:
    sub = panel_q.xs(entity, level='entity')
    r = converter.interpolate_to_higher_frequency(sub, 'MS', method='linear', limit=None, limit_area='inside')
    print(f"\n  entité {entity} ({r['pib'].isna().sum()} NaN restants) :")
    print(r.to_string(float_format='{:.2f}'.format))

In [ ]:
# Appliquer directement sur le panel (MultiIndex) sans dégrouper lève une erreur :
# resample/interpolate de pandas ne savent pas gérer un second niveau d'index
try:
    converter.aggregate_to_lower_frequency(panel_q, 'YE', method='mean')
except Exception as e:
    print(f"Panel passé directement (sans dégroupement par entité) -> {type(e).__name__} : {e}")
# C'est précisément pour cela que convert_frequency() route les panels vers
# _convert_panel_frequency, qui dégroupe avant d'appeler ces deux méthodes

## 10 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/frequency/test_converter.py` :

- **Bugs corrigés dans cette session** (à couvrir par des tests de non-régression) :
  - `get_conversion_factor()` levait un `NameError` avant correction (référençait
    des noms non définis) — vérifier `get_conversion_factor('daily', 'monthly') == 30.0`
    et la symétrie `factor(a,b) * factor(b,a) == 1`.
  - `interpolate_to_higher_frequency()` (et `convert_frequency()` en upsampling)
    levait un `AttributeError` dès que la base source différait de la base cible
    (ex. `Q -> M`) — vérifier explicitement une conversion trimestriel -> mensuel
    de bout en bout.
- **`convert()` == `convert_frequency()`** : `from_unit` n'est jamais utilisé ni
  validé (fréquence source toujours auto-détectée).
- **`get_conversion_factor()`** : symétrique (`factor(a,b) * factor(b,a) == 1`),
  identité (`factor(u,u) == 1`), accepte codes et littéraux indifféremment.
- **`count_subperiods_per_period()`** : comptage EXACT (28-31 jours/mois, 365/366
  jours/an) quand la base est convertible en `Period` pandas ; repli sur le
  facteur constant de `get_conversion_factor()` sinon (ex. `'SM'`).
- **`target_position=None`** ne préserve PAS toujours la position de la source
  contrairement à ce que suggère la docstring : pour une Series, seule la
  position explicitement présente dans la chaîne `target_freq` compte : une
  base nue (`'M'`) retombe sur la convention pandas historique (fin de
  période), quelle que soit la position de la source. Un point de vigilance à
  documenter et à figer par un test de régression.
- **`target_freq` en dict (DataFrame)** : l'index de sortie est l'UNION des
  index cibles des colonnes converties ; les colonnes absentes du dict (ou déjà
  à la fréquence cible) conservent leurs dates d'origine dans l'union ; les NaN
  introduits par l'union sont comblés selon `alignment_method`
  (`'ffill'`/`'bfill'`/`'nearest'`/`'none'`).
- **Panel (MultiIndex)** : `convert_frequency()` convertit chaque entité
  indépendamment sur son propre index simple (couverture temporelle et
  fréquence source propres à chaque entité) ; `target_freq` en dict accepte des
  clés `(entité,)`, `(entité, colonne)` ou `colonne`, avec la précédence
  `(entité, colonne)` > `(entité,)` > `colonne` ; les entités non ciblées
  gardent leur fréquence d'origine inchangée.
- **`aggregate_to_lower_frequency`** : `full_periods_only=True` masque (NaN) les
  périodes dont le nombre d'observations valides est inférieur au nombre de
  sous-périodes attendu ; sans cela, `sum` sous-estime silencieusement les
  périodes incomplètes. `method='all'`/`'any'` encodent leur propre sémantique
  de couverture (booléenne) et ignorent totalement `full_periods_only`.
- **`interpolate_to_higher_frequency`** : `limit='default'` utilise le facteur
  de conversion arrondi (3 pour `Q -> M`) ; `limit_direction` par défaut suit la
  position cible (`'forward'` pour start, `'backward'` pour end) ;
  `limit_area='inside'` évite toute extrapolation aux extrémités — recommandé
  pour des séries à délais de publication (NaN en début ET en fin).
- **`aggregate_to_lower_frequency`/`interpolate_to_higher_frequency` sur un
  panel MultiIndex directement** : lève une erreur (pandas ne sait pas
  `resample`/`interpolate` un second niveau d'index) — ces deux méthodes doivent
  être appliquées entité par entité, ce que fait `convert_frequency()` en
  interne via `_convert_panel_frequency`.
- **Fréquence source == cible** : `convert_frequency()` retourne les données
  inchangées (`.equals()` vrai), sans passer par `aggregate`/`interpolate`.
- **Erreurs de validation** : `target_freq` vide -> `ValueError` ; `target_freq`
  en dict sur une Series non-panel -> `ValueError`.